[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab02_Prompting.ipynb)

In [ ]:
#@title
print("""
╔═══════════════════════════════════════════════════════════╗
║                                                           ║
║         ██╗███╗   ██╗████████╗██╗     █████╗ ██╗        ║
║         ██║████╗  ██║╚══██╔══╝██║    ██╔══██╗██║        ║
║         ██║██╔██╗ ██║   ██║   ██║    ███████║██║        ║
║         ██║██║╚██╗██║   ██║   ██║    ██╔══██║██║        ║
║         ██║██║ ╚████║   ██║   ███████╗██║  ██║███████╗  ║
║         ╚═╝╚═╝  ╚═══╝   ╚═╝   ╚══════╝╚═╝  ╚═╝╚══════╝  ║
║                                                           ║
║               LAW + AI // LAB 02                          ║
║         Prompting the Algorithmic Leviathan             ║
║                                                           ║
║     Melbourne Law Masters 2026 // Session 2             ║
║                                                           ║
╚═══════════════════════════════════════════════════════════╝
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* — [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# Prompting the Algorithmic Leviathan

## Overview

In this session, we step out of the abstract and into the machine. You will interact directly with a Large Language Model (LLM) — specifically, Google's Gemini — and learn how it works **from the inside out**.

We will explore three core mechanisms:

1. **Tokenization**: How text is broken into discrete units (tokens) that the model actually processes
2. **Probability distributions**: How the model generates text by repeatedly predicting the next most likely token
3. **Stochasticity**: How randomness (temperature, sampling) shapes what the model produces

Then, we'll put the Leviathan to work on real international law questions. We'll ask it about the ICJ, self-defense, nuclear weapons, Palestine — and critically evaluate its reasoning. We'll hunt for hallucinations (fabricated case citations). We'll see what it gets right and, more importantly, what it gets spectacularly wrong.

The goal: understand not just *what* the model produces, but *why* — and therefore, when you can trust it and when you absolutely cannot.

---

**Prerequisites**: A valid Google Gemini API key (free tier available). Willingness to read code. A skeptical mind.

## Contents

> **[Part One: Setting Up — API Keys and First Contact](#scrollTo=part_one)**
>> Install dependencies and authenticate with Gemini

> **[Part Two: What Are Tokens? The Atoms of Language](#scrollTo=part_two)**
>> Tokenization, token IDs, and why the model sees chunks, not words

> **[Part Three: How Text Generation Actually Works](#scrollTo=part_three)**
>> Probability distributions, sampling, and the temperature dial

> **[Part Four: Testing the Leviathan on International Law](#scrollTo=part_four)**
>> Real questions. Real hallucinations. Real implications.

> **[Part Five: The Leviathan's Limits — Reflection](#scrollTo=part_five)**
>> Why the machine fails, and what this means for law

> **[Review Quiz](#scrollTo=review_quiz)**
>> Test your understanding

# Part One // Setting Up — API Keys and First Contact

In [ ]:
#@title ## Install Required Libraries

import subprocess
import sys

packages = ['google-genai', 'tiktoken', 'openai']

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✓ All dependencies installed!")

In [ ]:
#@title ## API Key Setup
#@markdown In Colab, secrets are stored securely. If you're running locally, use `getpass` instead.

import getpass
import os

def get_api_key(key_name, description):
    """Attempt to retrieve API key from Colab Secrets, fall back to getpass."""
    try:
        # Try Colab Secrets first
        from google.colab import userdata
        key = userdata.get(key_name)
        if key:
            print(f"✓ {description} loaded from Colab Secrets")
            return key
    except (ImportError, Exception):
        pass
    
    # Fall back to getpass
    print(f"Colab Secrets not available. Please enter your {description}:")
    return getpass.getpass(f"Enter {key_name}: ")

# Get Gemini API key (required)
google_api_key = get_api_key('GEMINI_API_KEY', 'Gemini API Key')

# Get OpenAI API key (optional)
try:
    openai_api_key = get_api_key('OPENAI_API_KEY', 'OpenAI API Key (optional)')
except:
    openai_api_key = None
    print("⚠ OpenAI API key not provided. Part Three demos will use Gemini only.")

print("\n✓ API keys configured!")

In [ ]:
#@title ## First Contact: Hello World

from google import genai
from google.genai import types

# Configure the Gemini client
client = genai.Client(api_key=google_api_key)

# Create a model instance
# Using client.models.generate_content() with model="gemini-3-flash-preview"

# Send a simple prompt
print("Sending prompt to Gemini...\n")
response = client.models.generate_content(model="gemini-3-flash-preview", contents=
    "Hello, Algorithmic Leviathan! Please introduce yourself in exactly one sentence."
)

print("Response from Gemini:")
print("-" * 60)
print(response.text)
print("-" * 60)
print("\n✓ Connection established! The Leviathan is awake.")

# Part Two // What Are Tokens? The Atoms of Language

## Tokenization: Breaking Language into Bits

Here's the secret: **Large Language Models do not "read" text the way you do.** They don't see words, sentences, or concepts. They see *tokens*.

A **token** is a small chunk of text — usually 3–4 characters on average, though it can be a whole word or even a punctuation mark. OpenAI's `tiktoken` tokenizer breaks English text into ~100K different tokens.

### Why This Matters

1. **Cost**: API pricing is often per-token. A 1,000-word essay isn't 1,000 tokens; it's more like 1,300–1,500.
2. **Context windows**: When we say an LLM has a "4,000-token context window," that's tokens, not words.
3. **What the model "understands"**: The model has learned patterns in token sequences. It has *no built-in understanding* of what tokens mean semantically.
4. **Bias and fairness**: Certain words (especially in non-English languages, or words from marginalized groups) get broken into more tokens, which affects how the model processes them.

Let's see this in action.

In [ ]:
#@title ## Tokenizing Legal Text

import tiktoken

# Use OpenAI's tokenizer (cl100k_base is used by GPT-3.5/GPT-4)
enc = tiktoken.get_encoding("cl100k_base")

# Sample sentences from international law
samples = [
    "The International Court of Justice held that...",
    "Article 51 of the UN Charter permits self-defense.",
    "ICJ advisory opinion: nuclear weapons are unlawful.",
    "Palestine seeks recognition as a sovereign state."
]

print("TOKENIZATION DEMO")
print("=" * 70)

for text in samples:
    tokens = enc.encode(text)
    print(f"\nText: '{text}'")
    print(f"Length (characters): {len(text)}")
    print(f"Tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
    print(f"Decoded: {[enc.decode_single_token_bytes(t) for t in tokens]}")
    print("-" * 70)

print("\n💡 KEY INSIGHT:")
print("Notice how 'International' is one token, but 'unlawful' is one token,")
print("and 'Palestine' is one token. Meanwhile, some words split across tokens.")
print("The model processes these token sequences, not the words themselves.")

In [ ]:
#@title ## Your Turn: Tokenize Any Text
#@markdown Enter any legal text below and see how it breaks into tokens.

student_text = "Enter your text here" #@param {type:"string"}

tokens = enc.encode(student_text)
print(f"Your text: '{student_text}'")
print(f"\nLength (characters): {len(student_text)}")
print(f"Tokens: {len(tokens)}")
print(f"\nToken IDs: {tokens}")
print(f"\nDecoded tokens:")
for i, token_id in enumerate(tokens):
    decoded = enc.decode_single_token_bytes(token_id).decode('utf-8', errors='replace')
    print(f"  Token {i}: {token_id:6d} → '{decoded}'")

print(f"\nAverage characters per token: {len(student_text) / len(tokens):.1f}")

# Part Three // How Text Generation Actually Works

## Next-Token Prediction: The Core Mechanism

Here's how an LLM generates text:

1. You give it a prompt (a sequence of tokens).
2. The model processes the entire sequence and outputs a **probability distribution** over all possible next tokens.
3. The model *samples* from this distribution (or takes the top token deterministically) to pick the next token.
4. It adds that token to the sequence and repeats.

### Temperature: The Randomness Dial

The **temperature** parameter controls how "creative" or "conservative" the model is:

- **Temperature = 0.0**: Deterministic. Always pick the highest-probability token. (Reproducible, safe, boring.)
- **Temperature = 0.5**: Balanced. Slight randomness, but favoring probable tokens.
- **Temperature = 1.0**: Standard. Full probability-weighted randomness.
- **Temperature = 2.0+**: Chaotic. Even low-probability tokens get selected. (Creative but often nonsensical.)

### Why This Matters

The model is **not reasoning** or **thinking**. It is **sampling from a learned probability distribution**. This has profound implications:

- It can hallucinate (assign high probability to tokens that don't correspond to real facts).
- It can give the same prompt different answers (if temperature > 0).
- It has no concept of truth—only likelihood in training data.

Let's see this in action.

In [ ]:
#@title ## Temperature Experiment: Same Prompt, Different Outputs
#@markdown Watch how the Leviathan's creativity shifts with temperature.

import time

prompt = "Draft the opening sentence of an International Court of Justice advisory opinion on the legality of artificial intelligence in warfare. Begin with 'The Court finds that'"

temperatures = [0.0, 0.5, 1.0, 1.5]

print("TEMPERATURE EXPERIMENT")
print("=" * 80)
print(f"Prompt: {prompt}\n")

for temp in temperatures:
    print(f"\n{'=' * 80}")
    print(f"Temperature: {temp}")
    print(f"{'=' * 80}")
    
    response = client.models.generate_content(model="gemini-3-flash-preview", contents=
        prompt,
        config=types.GenerateContentConfig(
            temperature=temp,
            max_output_tokens=100
        )
    )
    
    print(response.text)
    time.sleep(1)  # Be polite to the API

print(f"\n{'=' * 80}")
print("\n💡 OBSERVATION:")
print("At temperature=0.0, the response should be identical (deterministic).")
print("As temperature increases, outputs diverge and become more creative (and risky).")
print("\nFor legal reasoning, lower temperature is usually safer.")
print("For brainstorming, higher temperature can help explore ideas.")

In [ ]:
#@title ## Stochasticity Demo: Run the Same Prompt Multiple Times
#@markdown At temperature=0.7, run the same prompt 3 times. Notice the differences.

prompt_stochastic = "Name three landmark ICJ cases on state sovereignty."
temperature = 0.7

print(f"Prompt: '{prompt_stochastic}'")
print(f"Temperature: {temperature}")
print(f"\nRunning 3 times at the same temperature...\n")

for run in range(1, 4):
    print(f"\n--- RUN {run} ---")
    response = client.models.generate_content(model="gemini-3-flash-preview", contents=
        prompt_stochastic,
        config=types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=150
        )
    )
    print(response.text)
    time.sleep(1)

print("\n" + "=" * 80)
print("\n💡 KEY INSIGHT:")
print("Even with identical input and temperature, outputs differ!")
print("This is because the model samples from a probability distribution.")
print("\nFor reproducibility (e.g., in legal advice), use temperature=0.0.")

In [ ]:
#@title ## Your Turn: Temperature Experiment
#@markdown Write your own legal prompt and test it at different temperatures.

student_prompt = "What is the definition of self-defense under international law?" #@param {type:"string"}
student_temperature = 0.5 #@param {type:"slider", min:0, max:2, step:0.1}

print(f"Your prompt: '{student_prompt}'")
print(f"Temperature: {student_temperature}\n")

response = client.models.generate_content(model="gemini-3-flash-preview", contents=
    student_prompt,
    config=types.GenerateContentConfig(
        temperature=student_temperature,
        max_output_tokens=200
    )
)

print("Response:")
print("-" * 80)
print(response.text)
print("-" * 80)
print(f"\nNow try the same prompt with a different temperature above!")

# Part Four // Testing the Leviathan on International Law

## Putting the Leviathan to Work: Real International Law Questions

Now we ask the model real questions from your course. These are hard questions—questions where the answer involves nuance, competing interpretations, and serious legal reasoning.

For each answer, we'll analyze:
1. **What the model got right** (if anything)
2. **What the model got wrong** (if anything)
3. **Why the error occurred** (usually: the probability distribution favored a plausible but false token sequence)

We'll also hunt for **hallucinations**—cases where the model invents case citations that don't exist.

In [ ]:
#@title ## Question 1: Palestine and International Law

question_1 = "What is the legal status of the occupation of Palestine under international law? Cite relevant ICJ cases or UN resolutions."

print(f"QUESTION: {question_1}\n")
print("=" * 80)

response_1 = client.models.generate_content(model="gemini-3-flash-preview", contents=
    question_1,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=400
    )
)

print("RESPONSE:")
print(response_1.text)

### Analysis: Question 1

**What the model likely got right:**
- References to UN General Assembly resolutions and International Court of Justice advisory opinions
- The concept of illegal occupation under international law
- General principles from the UN Charter and Geneva Conventions

**What the model might have gotten wrong or overstated:**
- Specific case citations: Did it invent cases? Check against the ICJ official records
- Tone: Did it over-simplify or misrepresent contested legal questions?
- Missing nuance: International law scholars *disagree* on these points. The model may have presented one view as settled fact.

**Why errors occur:**
- The training data contains many plausible-sounding case names and citations.
- The model has learned that citing authority increases credibility (it's in the pattern of training data).
- But it has no mechanism to *verify* that citations are real—only to predict what tokens are likely to follow.

**Lesson:** When the model cites a case, *verify it yourself*. Do not trust citations from an LLM.

In [ ]:
#@title ## Question 2: Self-Defense and Non-State Actors

question_2 = "Can a state invoke self-defense against a non-state actor under Article 51 of the UN Charter? Discuss the Nicaragua case and subsequent developments."

print(f"QUESTION: {question_2}\n")
print("=" * 80)

response_2 = client.models.generate_content(model="gemini-3-flash-preview", contents=
    question_2,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=400
    )
)

print("RESPONSE:")
print(response_2.text)

### Analysis: Question 2

**What the model likely got right:**
- Article 51 of the UN Charter does permit self-defense
- The *Nicaragua v. United States* (1986) case is a real, landmark ICJ judgment
- The evolution of self-defense doctrine post-9/11 is a real development

**Critical issues:**
- **Nicaragua case**: The model should be accurate here since it's such a landmark case. But did it accurately state the holding? (The Court found that the US was not entitled to rely on self-defense against Nicaragua-supported rebels.)
- **Post-9/11 developments**: The model may conflate scholarly debate with settled law. Many states claim self-defense against non-state actors, but this remains legally contested.
- **Customary international law**: Did the model correctly describe whether self-defense against non-state actors has crystallized into customary law? (Answer: still debated.)

**Lesson:** The model is good at identifying *which* cases are relevant, but may misrepresent the *holding*. Always read the original judgment.

In [ ]:
#@title ## Question 3: Nuclear Weapons and International Law

question_3 = "Does customary international law prohibit the use of nuclear weapons? What did the ICJ say in its advisory opinion?"

print(f"QUESTION: {question_3}\n")
print("=" * 80)

response_3 = client.models.generate_content(model="gemini-3-flash-preview", contents=
    question_3,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=400
    )
)

print("RESPONSE:")
print(response_3.text)

### Analysis: Question 3

**What the model likely got right:**
- The ICJ did issue an advisory opinion on nuclear weapons in 1996
- The opinion is *Legality of the Threat or Use of Nuclear Weapons*
- The Court did NOT declare nuclear weapons categorically illegal

**Critical issue: The precise holding**
- The Court said that the use of nuclear weapons would be illegal *if it caused unnecessary suffering*, but the Court could not definitively rule that any use is prohibited in self-defense.
- Many commentators read this as: "nuclear weapons are implicitly illegal, but the Court avoided a direct statement."
- The model may oversimplify or misrepresent this careful (some would say evasive) holding.

**Hallucination risk:** Did the model cite other cases on nuclear weapons? Verify any citations.

**Lesson:** Even with a case the model can identify correctly, it may misrepresent the nuanced holding. The ICJ's nuclear weapons opinion is famously ambiguous—and the model may flatten that ambiguity into false certainty.

In [ ]:
#@title ## Hallucination Hunt: Fabricated Case Citations
#@markdown Ask the Leviathan to cite cases. Then fact-check them.

hallucination_prompt = "List five ICJ cases on the use of force, with the full case name, year decided, and a one-sentence summary of the holding."

print(f"PROMPT: {hallucination_prompt}\n")
print("=" * 80)

response_hallucination = client.models.generate_content(model="gemini-3-flash-preview", contents=
    hallucination_prompt,
    config=types.GenerateContentConfig(
        temperature=0.3,
        max_output_tokens=500
    )
)

print("RESPONSE:")
print(response_hallucination.text)
print("\n" + "=" * 80)
print("\n⚠️  HALLUCINATION HUNT:")
print("\nYour task: For each case listed above, fact-check it.")
print("- Visit the ICJ website (www.icj-cij.org)")
print("- Search for the case name")
print("- Does it exist? If so, is the year and summary correct?")
print("\nMost likely outcome: At least one case will be fabricated.")
print("This is not a bug in the model—it is a structural feature.")
print("The model samples from a probability distribution. When tokens")
print("representing plausible-sounding case names have high probability,")
print("the model will output them even if they don't correspond to real cases.")

In [ ]:
#@title ## Your Turn: Ask Your Own Question
#@markdown Write a question about international law. Test the Leviathan. Evaluate its answer.

student_question = "What is the principle of non-intervention in international law?" #@param {type:"string"}
student_temp = 0.5 #@param {type:"slider", min:0, max:1.5, step:0.1}

print(f"Your question: {student_question}")
print(f"Temperature: {student_temp}\n")
print("=" * 80)

response_student = client.models.generate_content(model="gemini-3-flash-preview", contents=
    student_question,
    config=types.GenerateContentConfig(
        temperature=student_temp,
        max_output_tokens=300
    )
)

print("RESPONSE:")
print(response_student.text)
print("\n" + "=" * 80)
print("\n📋 REFLECTION QUESTIONS:")
print("1. What did the model get right?")
print("2. What seems uncertain or potentially wrong?")
print("3. Did the model cite any cases? If so, can you verify them?")
print("4. How would you fact-check this answer? (Hint: read actual court documents.)")

# Part Five // The Leviathan's Limits — Reflection

## Why the Machine Fails: A Structural Analysis

Let's be clear about what we've seen:

### 1. **Hallucination Is Inevitable**

Hallucination—the generation of plausible-sounding but false information—is not a bug. It is a structural feature of how LLMs work.

Why? Because the model is sampling from a probability distribution. When a token sequence (like "*Case v. Country* (Year)") has high probability in the training data (because many real cases follow this pattern), the model will output it even if no real case with exactly those names exists.

**Example**: The model has seen millions of case citations in training. It has learned that case citations are formatted like "[Plaintiff] v. [Defendant], [Court] ([Year])." When asked to cite five cases, it will generate text matching this pattern—but some may be entirely fabricated.

### 2. **No Distinction Between Fact and Pattern**

The model has learned patterns in the training data, not facts about the world. It has no mechanism to distinguish between:
- "The ICJ is in The Hague" (true, and in many training texts)
- "The ICJ sits in Geneva" (false, but generated if the token sequence has probability)

Both are just token sequences. The model assigns higher probability to the first because it appeared more often in training data—but this is *statistical*, not semantic.

### 3. **No Real Understanding of Law**

When the model discusses "self-defense under Article 51," it is not *reasoning* about the law. It is predicting which tokens are most likely to follow "self-defense under Article 51" based on patterns in legal texts.

It has zero understanding of:
- What "self-defense" *means*
- Why lawyers *argue* about it
- How judges *reason* about it
- What difference a given interpretation makes to real people

It has learned to produce text that *looks* like it understands these things.

### 4. **The Stochastic Parrot Critique (Bender et al., 2021)**

Computer scientists Emily Bender, Timnit Gebru, and others published a paper called *On the Dangers of Stochastic Parrots* that makes this point sharply:

> LLMs are "stochastic parrots"—they are not learning language; they are learning to produce text that statistically resembles human language. They have no grounding in the world, no genuine understanding, and no ability to reason.

This paper caused uproar in the AI industry. But for lawyers, it should ring true. You already know: *mimicking legal reasoning is not reasoning*. A law student who memorizes case citations but doesn't understand the principles is not practicing law—they're performing law.

### 5. **So What? Can We Still Use LLMs in Law?**

Yes—but carefully.

**Safe uses:**
- Summarizing and organizing documents you've already reviewed
- Brainstorming arguments (then fact-checking each one)
- Identifying relevant cases to look up (but verifying each one)
- Drafting boilerplate (with careful review)

**Dangerous uses:**
- Relying on citations without verification
- Treating model outputs as authoritative legal advice
- Using the model to answer novel questions without human expert review
- Assuming the model will flag its own uncertainty (it won't; it will hallucinate with confidence)

**The bottom line**: LLMs are tools for *research assistance*, not for *decision-making*. Think of them like a research librarian who is incredibly fast but sometimes makes up book titles. Useful—but not trustworthy without verification.

---

### Further Reading

- **Bender, E. M., Gebru, T., et al. (2021).** "On the Dangers of Stochastic Parrots." *Proceedings of FAccT '21*.
- **Cant, A. (2024).** *How to Survive the Age of AI*. [Relevant sections on AI and knowledge work]
- **AJIL Unbound articles** on AI and international law (see course readings)

These readings connect to later sessions in this course, where we'll discuss the legal, ethical, and institutional implications of AI in governance.

# Review Quiz

In [ ]:
#@title ## Quiz Q1: What is a Token?

q1_answer = "A small chunk of text (typically 3-4 characters on average) that the model processes" #@param {type:"string", options:["A small chunk of text (typically 3-4 characters on average) that the model processes", "A word in the text", "A sentence in the text", "A semantic concept the model understands"]}

if "chunk" in q1_answer.lower():
    print("✓ CORRECT! Tokens are the atomic units. 'International' might be one token, but 'internationalization' might be split into multiple tokens.")
else:
    print("✗ Not quite. Tokens are smaller than words and the model doesn't 'understand' them semantically. It just predicts token sequences.")
    print(f"Your answer: {q1_answer}")

In [ ]:
#@title ## Quiz Q2: What Does Temperature Control?

q2_answer = "How random/creative the model's output is" #@param {type:"string", options:["How random/creative the model's output is", "How fast the model generates text", "The size of the model", "The accuracy of the model's citations"]}

if "random" in q2_answer.lower() or "creative" in q2_answer.lower():
    print("✓ CORRECT! Temperature=0 is deterministic; temperature > 1 is creative and risky.")
else:
    print("✗ Not quite. Temperature controls the sampling distribution. Higher temperature = more randomness.")
    print(f"Your answer: {q2_answer}")

In [ ]:
#@title ## Quiz Q3: Why Do LLMs Hallucinate?

q3_answer = "Because they sample from a probability distribution and have no mechanism to verify facts" #@param {type:"string", options:["Because they sample from a probability distribution and have no mechanism to verify facts", "Because they are broken and need more training", "Because they are evil and want to deceive us", "Because they don't have enough parameters"]}

if "probability" in q3_answer.lower() or "verify" in q3_answer.lower():
    print("✓ CORRECT! Hallucination is structural. When plausible-sounding tokens have high probability, the model outputs them—even if false.")
else:
    print("✗ Not quite. Hallucination isn't a bug or moral failing. It's how sampling from probability distributions works.")
    print(f"Your answer: {q3_answer}")

In [ ]:
#@title ## Quiz Q4: Temperature 0.0 vs. 1.0

q4_answer = "0.0 is deterministic (same output every time); 1.0 is stochastic (different outputs)" #@param {type:"string", options:["0.0 is deterministic (same output every time); 1.0 is stochastic (different outputs)", "0.0 is creative; 1.0 is boring", "0.0 is accurate; 1.0 is hallucinating", "There is no difference"]}

if "deterministic" in q4_answer.lower() or "stochastic" in q4_answer.lower():
    print("✓ CORRECT! At temperature=0.0, the model always picks the highest-probability token (reproducible). At 1.0, it samples more randomly.")
else:
    print("✗ Not quite. Temperature changes the sampling behavior, not the quality or creativity in a simple way.")
    print(f"Your answer: {q4_answer}")

In [ ]:
#@title ## Quiz Q5: Can an LLM Really Know International Law?

q5_answer = "No—it learns patterns in text, not semantic understanding of law" #@param {type:"string", options:["No—it learns patterns in text, not semantic understanding of law", "Yes—if trained on enough legal documents", "Maybe—we need more research to know", "Yes—the 'reasoning' cells emerge at scale"]}

if "patterns" in q5_answer.lower() or "not semantic" in q5_answer.lower():
    print("✓ CORRECT! The model has learned what text about international law looks like, not what international law *is*.")
else:
    print("✗ Not quite. This is the core insight: LLMs are stochastic text predictors, not reasoners.")
    print(f"Your answer: {q5_answer}")

## Conclusion: The Algorithmic Leviathan

In this lab, you've seen inside the machine. You've learned:

1. **Tokenization**: LLMs process text as tokens, not words. This shapes cost, context, and capability.
2. **Next-token prediction**: Text generation is sampling from a learned probability distribution, not reasoning.
3. **Temperature**: You can control randomness—but not accuracy.
4. **Hallucination**: Inevitable, structural, and dangerous in legal contexts.
5. **The stochastic parrot**: Mimicking legal reasoning is not reasoning.

As we move forward in this course, you'll see how these facts shape:

- **Session 7**: Accountability and AI in legal systems
- **Session 9**: Regulation and governance
- **Final project**: Designing safeguards for AI in law

For now: **Be skeptical. Verify. Never trust a citation from an LLM without checking it yourself.**

---

**Next session**: We'll dive into the legal frameworks—what international law *already* says about AI, and where the gaps are.

Until then: Prompt responsibly. 🔬